In [34]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset,DataLoader,Subset
import kagglehub

In [35]:
torch.manual_seed(42)

In [36]:
device = torch.device('cuda'if torch.cuda.is_available() else 'cpu')

In [37]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'plantvillage' dataset.
Path to dataset files: /kaggle/input/plantvillage


In [38]:
TRAIN_PATH = os.path.join(path,"PlantVillage","train")
VAL_PATH = os.path.join(path,"PlantVillage","val")

In [39]:
transform = transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ]
)

In [40]:
from genericpath import isfile
class  MultiClassClassification(Dataset):
  def __init__(self, root_dir,transform=None):
    super().__init__()
    self.samples = []
    self.transform = transform
    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])
    self.class_to_idx = {cls_name: idx for idx,cls_name in enumerate(self.classes)}
    for class_name in self.classes:
      class_path = os.path.join(root_dir,class_name)

      for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path,img_name)
        if  os.path.isfile(img_path):
          label = self.class_to_idx[class_name]
          self.samples.append((img_path,label))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    img_path,label = self.samples[idx]
    image = Image.open(img_path).convert("RGB")

    if self.transform:
      image = self.transform(image)
    return image,label

In [41]:
train_dataset_full = MultiClassClassification(TRAIN_PATH,transform)
test_dataset_full = MultiClassClassification(VAL_PATH,transform)
num_classes = len(train_dataset_full.classes)

In [42]:
print("Number of classes:",num_classes)
print("Full train size:", len(train_dataset_full))
print("Full test size:", len(test_dataset_full))

Number of classes: 38
Full train size: 43444
Full test size: 10861


In [43]:
# train_dataset = Subset(train_dataset_full,list(range(min(100,len(train_dataset_full)))))
# test_dataset = Subset(test_dataset_full,list(range(min(100,len(train_dataset_full)))))

In [44]:
train_loader = DataLoader(train_dataset_full,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset_full,batch_size=32,shuffle=False,pin_memory=True)

In [45]:
class MyCNN(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(2),
        nn.Conv2d(32,64,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(2),
        nn.Conv2d(64,128,kernel_size=3,padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(128*16*16,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(64,num_classes)
    )

  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [46]:
model = MyCNN(num_classes=num_classes).to(device)

In [47]:
learning_rate = 0.001
ephocs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=learning_rate)

In [48]:
for epoch in range(ephocs):
  model.train()
  total_loss = 0
  for batch_features,batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    loss = criterion(outputs,batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  avg_loss = total_loss/len(train_loader)
  print(f"Epoch {epoch+1}/{ephocs}, Loss:{avg_loss:.4f}")

Epoch 1/10, Loss:1.9977
Epoch 2/10, Loss:1.3081
Epoch 3/10, Loss:1.0711
Epoch 4/10, Loss:0.9287
Epoch 5/10, Loss:0.7687
Epoch 6/10, Loss:0.6673
Epoch 7/10, Loss:0.5835
Epoch 8/10, Loss:0.5253
Epoch 9/10, Loss:0.4745
Epoch 10/10, Loss:0.4425


In [49]:
model.eval()

MyCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (9): ReLU()
    (10): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, 

In [50]:
total,correct = 0,0
with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    _,predicted = torch.max(outputs,1)
    total += batch_labels.size(0)
    correct += (predicted == batch_labels).sum().item()
  print(correct/total)

0.938679679587515
